In [ ]:
import pandas as pd
df = pd.read_csv('password_data.csv',on_bad_lines='skip')
df.head()

In [ ]:

df = df.dropna()
df = df.drop_duplicates(subset=['password'])
print("Data pre-processing completed successfully!")
print("Final dataset shape (Rows, Columns):", df.shape)

In [ ]:
import string
def get_length(password):
    return len(str(password))
def count_uppercase(password):
    return sum(1 for c in str(password) if c.isupper())

def count_lowercase(password):
    return sum(1 for c in str(password) if c.islower())

def count_digits(password):
    return sum(1 for c in str(password) if c.isdigit())

def count_special(password):
    return sum(1 for c in str(password) if c in string.punctuation)

df['length'] = df['password'].apply(get_length)
df['uppercase'] = df['password'].apply(count_uppercase)
df['lowercase'] = df['password'].apply(count_lowercase)
df['digits'] = df['password'].apply(count_digits)
df['special_chars'] = df['password'].apply(count_special)

print("--- Features extracted successfully ---")
print(df[['password', 'length', 'uppercase', 'lowercase', 'digits', 'special_chars']].head())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import joblib

# 1. Separate features (X) and target/label (y)
# X contains the numbers we calculated, y contains the actual strength (0, 1, 2)
X = df[['length', 'uppercase', 'lowercase', 'digits', 'special_chars']]
y = df['strength']

# 2. Split data into training set (80%) and testing set (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")

# 3. Initialize the Random Forest Classifier model
model = RandomForestClassifier(n_estimators=100, random_state=42)

# 4. Train (Fit) the model on training data
print("\nModel training started... (This might take a moment)")
model.fit(X_train, y_train)
print("Model training completed successfully! 🎉")

# 5. Save the trained model as a .pkl file for Power BI and Future deployment
joblib.dump(model, 'password_strength_model.pkl')
print("Model saved as 'password_strength_model.pkl'")

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Make predictions on the hidden test data
y_pred = model.predict(X_test)

# 2. Calculate and print the overall accuracy of the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Overall Model Accuracy: {accuracy * 100:.2f}%\n")

# 3. Print a detailed evaluation report (Precision, Recall, F1-Score)
print("--- Detailed Classification Report ---")
print(classification_report(y_test, y_pred, target_names=['Weak (0)', 'Medium (1)', 'Strong (2)']))

In [ ]:
import string

# Function to transform a single new password into the 5 features
def check_live_password(new_password):
    length = len(str(new_password))
    uppercase = sum(1 for c in str(new_password) if c.isupper())
    lowercase = sum(1 for c in str(new_password) if c.islower())
    digits = sum(1 for c in str(new_password) if c.isdigit())
    special_chars = sum(1 for c in str(new_password) if c in string.punctuation)
    
    # Put features into the exact order the model expects
    features = [[length, uppercase, lowercase, digits, special_chars]]
    
    # Predict using our trained model
    prediction = model.predict(features)[0]
    
    # Map the numeric result to a text label
    labels = {0: "Weak (0)", 1: "Medium (1)", 2: "Strong (2)"}
    return labels[prediction]

# Let's test the model with 3 completely different custom passwords
print("Test 1 ('12345'):", check_live_password("12345"))
print("Test 2 ('Abcd123'):", check_live_password("Abcd123"))
print("Test 3 ('P@ssW0rd_2026!'):", check_live_password("P@ssW0rd_2026!"))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import pandas as pd
import numpy as np

# 1. Create a Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Weak', 'Medium', 'Strong'], 
            yticklabels=['Weak', 'Medium', 'Strong'])
plt.title('Confusion Matrix of Password Model')
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.show()

# 2. Plot Feature Importance (Which feature matters the most?)
importances = model.feature_importances_
feature_names = ['length', 'uppercase', 'lowercase', 'digits', 'special_chars']

# Store features and their importance scores in a DataFrame
feat_importances = pd.Series(importances, index=feature_names).sort_values(ascending=True)

plt.figure(figsize=(8, 4))
feat_importances.plot(kind='barh', color='teal')
plt.title('Feature Importance for Password Strength')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.show()

In [ ]:
# Save the cleaned dataset with features to a new CSV file for Power BI
df.to_csv('cleaned_password_data.csv', index=False)
print("Data saved successfully as 'cleaned_password_data.csv' for Power BI!")

In [ ]:
import string

# Clear instructions for the user with the updated 16-length rule
print("--- Final Password Strength Checker (Max Length: 16) ---")
print("Note: Password length CANNOT be more than 16 characters.")
print("Type 'exit' to stop the program.")

while True:
    # 1. Take dynamic input from the user continuously
    user_password = input("\nEnter a password to check: ")
    
    # 2. Check if the user wants to stop the program
    if user_password.lower() == 'exit':
        print("Program closed. Thank you! 👋")
        break  # Stop the loop
        
    # 3. Calculate the length of the input password
    length = len(str(user_password))
    
    # 4. Strict Check: If length is greater than 16, show error and ask again
    if length > 16:
        print(f"❌ Error: Your password length is {length}. Maximum allowed length is 16 characters!")
        continue  # Skip prediction and restart loop
        
    # 5. Extract features only if length is within the limit (<= 16)
    uppercase = sum(1 for c in str(user_password) if c.isupper())
    lowercase = sum(1 for c in str(user_password) if c.islower())
    digits = sum(1 for c in str(user_password) if c.isdigit())
    special_chars = sum(1 for c in str(user_password) if c in string.punctuation)
    
    # 6. Format features into the exact shape the model expects
    features = [[length, uppercase, lowercase, digits, special_chars]]
    
    # 7. Predict using the trained model
    prediction = model.predict(features)
    
    # 8. Display the final strength result
    if prediction == 0:
        print(f"Result: The password '{user_password}' is WEAK (0). 🔴")
    elif prediction == 1:
        print(f"Result: The password '{user_password}' is MEDIUM (1). 🟡")
    else:
        print(f"Result: The password '{user_password}' is STRONG (2). 🟢")